In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ENTREPOT_PATH = "/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/"
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'composant_culture',
    'culture',
    'espece'
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

100%|██████████| 3/3 [00:03<00:00,  1.13s/it]


In [3]:
culture = donnees['culture'].reset_index().rename(columns={'id':'culture_id'}).copy()
cc = donnees['composant_culture'].reset_index().rename(columns={'id':'composant_culture_id'}).copy()
# espece = donnees['espece'].rename(columns={'id':'espece_id'}).reset_index().copy()


espece = pd.read_csv("/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/data/external_data/"+"espece.csv", sep=',')

MAE_matrice_precise = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_matrice_passage_dirodur_espece_precise.csv').rename(columns={'id':'espece_id'})

MAE_matrice_famille = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_matrice_passage_dirodur_famille_bota.csv').rename(columns={'id':'espece_id'})


with open('/home/administrateur/Bureau/' + 'liste_culture_id_gcpe.txt', "r") as f:
    list_cid_gcpe = [line.strip() for line in f.readlines()]

In [4]:
df = cc.merge(espece.rename(columns={'id': 'espece_id'}), how='left', on='espece_id')
# On ne garde que les culture_id qui ont été utilisées dans un sdc GCPE
culture = culture.loc[culture['culture_id'].isin(list_cid_gcpe)]
df = df.merge(culture, how='right', on='culture_id')

In [5]:
def concat_unique_sorted(series):
    cleaned = series.dropna().unique()
    if len(cleaned) == 0:
        return np.nan
    return '_'.join(sorted(cleaned))
def get_nb_unique_typo(series):
    cleaned = series.dropna().unique()
    return len(cleaned)

In [6]:
agg_dict = {
    'nb_composant_culture': 'sum',
    'typodirodur_espece_precise': concat_unique_sorted,
    'typodirodur_espece_famille_bota': concat_unique_sorted,
    'typodirodur_espece_periode_semis': concat_unique_sorted,
    'nb_typodirodur_espece_precise': get_nb_unique_typo,
    'nb_typodirodur_espece_famille_bota': get_nb_unique_typo,
}

In [7]:
len(df.loc[df['composant_culture_id'].isna(), 'nom'].unique().tolist())

361

In [8]:
df['nb_composant_culture'] = 1
df['nb_typodirodur_espece_precise'] = df['typodirodur_espece_precise']
df['nb_typodirodur_espece_famille_bota'] = df['typodirodur_espece_famille_bota']

df2 = df[['culture_id',
         'nb_composant_culture',
         'typodirodur_espece_precise',
         'typodirodur_espece_famille_bota',
         'typodirodur_espece_periode_semis',
         'nb_typodirodur_espece_precise',
         'nb_typodirodur_espece_famille_bota']].groupby('culture_id').agg(agg_dict).reset_index()

In [9]:
df2.typodirodur_espece_precise.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_precise
Blé tendre Hiver Meunier                                                                                                                   9269
Maïs                                                                                                                                       9134
Blé tendre Hiver Biscuit                                                                                                                   6860
Colza Oléagineux Hiver                                                                                                                     6741
Orge 6 rangs Hiver                                                                                                                         4741
                                                                                                                                           ... 
Avoine Noir(e) Hiver_Blé tendre Hiver Amidon_Orge 2 rangs Hiver_Pois Fourrager / Fourrage Hiver_Triticale_Ves

In [10]:
df2.typodirodur_espece_famille_bota.value_counts(dropna=False).sort_values(ascending=False)

typodirodur_espece_famille_bota
Poaceae                                                 50922
Fabaceae                                                12594
Brassicaceae                                             8645
Fabaceae_Poaceae                                         8005
NaN                                                      6089
                                                        ...  
Asteraceae_Fabaceae_Poaceae_Rubiaceae                       1
Boraginaceae                                                1
Renonculaceae                                               1
Asphodelaceae_Asteraceae_Poaceae                            1
Asteraceae_Brassicaceae_Hydrophyllaceae_Polygonaceae        1
Name: count, Length: 151, dtype: int64

In [11]:
df3 = df2.loc[df2['nb_typodirodur_espece_precise'] != 1]
df3 = df3.groupby(['typodirodur_espece_precise','typodirodur_espece_periode_semis']).size().sort_values(ascending=False).reset_index()
df3.rename(columns={0:'count'}, inplace=True)
df3.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_espece_precise_v2.csv", index=False)
df3

,typodirodur_espece_precise,typodirodur_espece_periode_semis,count
0,Pois Fourrager / Fourrage Hiver_Triticale,hiver,282
1,Fétuque élevée_Ray-grass anglais_Trèfle blanc,pluriannuelle,245
2,Blé tendre Hiver Amidon_Blé tendre Hiver Meunier,hiver,227
3,Blé tendre Hiver Meunier_Fèverole Hiver,hiver,201
4,Ray-grass anglais_Trèfle blanc,pluriannuelle,182
...,...,...,...
3135,Autres ornementales_Carthame_Fenugrec_Phacélie...,ete,1
3136,Autres ornementales_Carthame_Fenugrec_Radis Da...,ete,1
3137,Avoine Blanche Hiver_Avoine Noir(e) Hiver_Fève...,hiver_pluriannuelle,1
3138,Avoine Blanche Hiver_Avoine Noir(e) Hiver_Fève...,hiver,1


In [13]:
MAE_espece_precise = pd.read_csv('/home/administrateur/Bureau/' + 'MAE_matrice_passage_dirodur_espece_precise.csv')

df3_v2 = MAE_espece_precise.merge(df3, how='outer', on='typodirodur_espece_precise')

df3_v2.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_espece_precise_v3.csv", index=False)

In [ ]:
df4 = df2.loc[df2['nb_typodirodur_espece_famille_bota'] != 1]
df4 = df4.groupby(['typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']).size().sort_values(ascending=False).reset_index()
df4.rename(columns={0:'count'}, inplace=True)
df4

In [ ]:
df3 = df3.merge(MAE_matrice_precise[['typodirodur_espece_precise', 
                     'typodirodur_culture',
                     'culture_avec_compagne', 
                     'culture_annuelle_asso', 
                     'prairie']], on='typodirodur_espece_precise', how='left')
df3

In [ ]:
# df4 = df4.merge(MAE_matrice_precise[['typodirodur_espece_famille_bota', 
#                      'typodirodur_culture',
#                      'culture_avec_compagne', 
#                      'culture_annuelle_asso', 
#                      'prairie']], on='typodirodur_espece_famille_bota', how='left')

# df4['duplic_date_semis'] = np.nan
# df4.loc[df4.duplicated(subset=['typodirodur_espece_famille_bota'], keep=False), 'duplic_semis'] = 'check_date_semis'

# df4.to_csv('/home/administrateur/Bureau/' + "matrice_passage_dirodur_famille_bota_v2.csv", index=False)
# df4